## This notebook contains all of steps used to wragle and clean the EEOC data for my Exploratory Data Analysis.

When I orginally embarked on this jounrey, everytime I tried something for the sake of expirementation, I would save it in a new file instead of keeping it all in one place. My goal here is to walk you through all of the steps that reulted in my final Exploratory Data Analysis by including all of the different scripts in this one file. 


### 1. Obtaining the EEOC Data

The EEOC Datasets were acquired through the [EEOC Government Website](https://www.eeoc.gov/data/employment-statistics). These datasets are saved in the directory folder titled `EEOC_Data`.

Within `EEOC_Data`, there are 4 subfolders, with each containing `.csv` or `.xlsx` versions of the dataset. Each folder represents a different EEOC dataset as defined below:

- **EEOC_1 Data:** Private Employers
- **EEOC_3 Data:** Referral Unions
- **EEOC_4 Data:** State and Local Government
- **EEOC_5 Data:** Elementary and Secondary Schools

> **Note:** There is no EEOC_2 reporting data, as that reporting was consolidated into EEOC-1 reporting in the 1970s.

I focus soley on `EEOC_1_Data` from here on out. This data was more than suffecient for my first time wrangling, cleaning, and performing an EDA. 

### 2. Script Creation for Downloading and Saving EEOC Datasets

This is the script I used to standardize the names of my downloaded EEOC_1 datasets. I decided to download the individual EEOC_1_Data Sets for the years 2014-2023. 

In [1]:
import os 
import re # I'm following Gemini's advice to import this function

def standardize_excel_puf_files(directory_path):
    """
    Scans a directory for EEOC PUF files and renames them to EEOC1_YYYY.[original_extension]
    """
    if not os.path.exists(directory_path):
        print(f"Error: The directory '{directory_path}' does not exist.")
        return

    print(f"Scanning directory: {directory_path}\n" + "-"*30)

    # Regex looks for 'EEO', the 4-digit year, and 'PUF' anywhere in the name
    pattern = re.compile(r'EEO.*?(\d{4}).*?PUF', re.IGNORECASE)

    for filename in os.listdir(directory_path):
        old_filepath = os.path.join(directory_path, filename)

        if os.path.isfile(old_filepath):
            match = pattern.search(filename)
            
            if match:
                year = match.group(1)
                
                # Extract the actual extension (.xls or .xlsx)
                name, ext = os.path.splitext(filename)
                
                # Construct the new filename (e.g., EEOC1_2017.xlsx)
                new_filename = f"EEOC1_{year}{ext.lower()}"
                new_filepath = os.path.join(directory_path, new_filename)

                # Rename the file safely
                if old_filepath != new_filepath:
                    if not os.path.exists(new_filepath):
                        os.rename(old_filepath, new_filepath)
                        print(f"Renamed: '{filename}'  ->  '{new_filename}'")
                    else:
                        print(f"Skipped: '{new_filename}' already exists.")

# ==========================================
# How to use it:
# Replace the string below with the path to your folder.
# ==========================================

target_folder = os.path.join("EEOC_Data", "EEOC_1_Data")
standardize_excel_puf_files(target_folder) # Remove the '#' to run

Scanning directory: EEOC_Data\EEOC_1_Data
------------------------------


Then I created this script to change all of the EEOC_1 excel files into .csv format. 

The source script is saved as `Change_EEOC_File_Type.ipynb`

In [2]:
print("Starting the script...", flush=True)

print("Loading pandas (this might take a moment)...", flush=True)
import os
import pandas as pd
print("Pandas loaded successfully!", flush=True)

def convert_standardized_eeoc_to_csv(directory_path):
    if not os.path.exists(directory_path):
        print(f"Error: The directory '{directory_path}' does not exist.", flush=True)
        return

    print(f"Scanning directory: {directory_path} for EEOC1 Excel files...\n" + "-"*40, flush=True)

    converted_count = 0

    for filename in os.listdir(directory_path):
        old_filepath = os.path.join(directory_path, filename)

        if os.path.isfile(old_filepath) and filename.startswith('EEOC1_') and filename.endswith(('.xls', '.xlsx')):
            
            name, ext = os.path.splitext(filename)
            new_filename = f"{name}.csv"
            new_filepath = os.path.join(directory_path, new_filename)

            if not os.path.exists(new_filepath):
                print(f"Converting: '{filename}'  ->  '{new_filename}'...", flush=True)
                
                try:
                    engine = 'openpyxl' if ext.lower() == '.xlsx' else 'xlrd'
                    df = pd.read_excel(old_filepath, engine=engine)
                    df.to_csv(new_filepath, index=False)
                    
                    print("  Success!", flush=True)
                    converted_count += 1
                    
                except Exception as e:
                    print(f"  Error converting '{filename}': {e}", flush=True)
            else:
                print(f"Skipped: '{new_filename}' already exists.", flush=True)

    print("\n" + "-"*40, flush=True)
    print(f"Process complete! Converted {converted_count} files.", flush=True)


# ==========================================
# PASTE YOUR FOLDER PATH HERE:
target_folder = os.path.join("EEOC_Data", "EEOC_1_Data") 

# MAKE SURE THERE IS NO '#' AT THE START OF THE NEXT LINE:
convert_standardized_eeoc_to_csv(target_folder)

Starting the script...
Loading pandas (this might take a moment)...
Pandas loaded successfully!
Scanning directory: EEOC_Data\EEOC_1_Data for EEOC1 Excel files...
----------------------------------------
Skipped: 'EEOC1_2014.csv' already exists.
Skipped: 'EEOC1_2015.csv' already exists.
Skipped: 'EEOC1_2016.csv' already exists.
Skipped: 'EEOC1_2017.csv' already exists.
Skipped: 'EEOC1_2018.csv' already exists.
Skipped: 'EEOC1_2019.csv' already exists.
Skipped: 'EEOC1_2020.csv' already exists.
Skipped: 'EEOC1_2021.csv' already exists.
Skipped: 'EEOC1_2022.csv' already exists.
Skipped: 'EEOC1_2023.csv' already exists.

----------------------------------------
Process complete! Converted 0 files.


### 3. Data Cleaning Scripts

`EEOC1_Exploration.ipynb` is the first script I created to get my first looks at an EEOC_1 Dataset

The scripts below were made to clean the datasets. 
- Added a 'YEAR' column to each one dataset.
- Dropped most of the aggregated columns, reducing the datasets from 275 columns to 22. 
    > **Note:** This ended up being a mistake and I had to go back and change this in a later script. I thought I would be able to create my own aggregates if I wanted to later on. 

`EEOC1_Exploration.ipynb` Script:

In [3]:
import pandas as pd
import dtale
import numpy as np

EEOC1_2023 = pd.read_csv('EEOC_Data/EEOC_1_Data/EEOC1_2023.csv')

EEOC1_2023.info()

C:\Users\jfraz\AppData\Local\Temp\ipykernel_82268\1732477512.py:5: DtypeWarning: Columns (0: Region, 1: Division, 2: State, 3: County) have mixed types. Specify dtype option on import or set low_memory=False.
  EEOC1_2023 = pd.read_csv('EEOC_Data/EEOC_1_Data/EEOC1_2023.csv')


<class 'pandas.DataFrame'>
RangeIndex: 105613 entries, 0 to 105612
Columns: 275 entries, Nation to TOMRF1_2
dtypes: float64(2), int64(2), str(271)
memory usage: 221.6 MB


In [4]:
EEOC1_2023.head()

,Nation,Region,Division,State,CBSA,County,NAICS2,NAICS2_Name,NAICS3,NAICS3_Name,...,AIANM1_2,NHOPIM1_2,TOMRM1_2,WHF1_2,BLKF1_2,HISPF1_2,ASIANF1_2,AIANF1_2,NHOPIF1_2,TOMRF1_2
0,United States,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,18884,18537,94570,2341332,375584,418200,267220,16434,16937,90549
1,United States,NaN,NaN,NaN,NaN,NaN,11.0,"Agriculture, Forestry, Fishing and Hunting",NaN,NaN,...,119,42,217,4789,504,1995,176,47,19,133
2,United States,NaN,NaN,NaN,NaN,NaN,11.0,"Agriculture, Forestry, Fishing and Hunting",111.0,Crop Production,...,51,22,111,*,106,*,90,27,13,54
3,United States,NaN,NaN,NaN,NaN,NaN,11.0,"Agriculture, Forestry, Fishing and Hunting",112.0,Animal Production and Aquaculture,...,58,8,57,1996,346,487,59,13,5,51
4,United States,NaN,NaN,NaN,NaN,NaN,11.0,"Agriculture, Forestry, Fishing and Hunting",113.0,Forestry and Logging,...,*,*,*,107,5,6,*,*,*,*


In [5]:
dtale.show(EEOC1_2023, open_browser=True)

In [6]:
EEOC1_2023.info()

<class 'pandas.DataFrame'>
RangeIndex: 105613 entries, 0 to 105612
Columns: 275 entries, Nation to TOMRF1_2
dtypes: float64(2), int64(2), str(271)
memory usage: 221.6 MB


In [7]:
EEOC1_2023.describe(
)

,NAICS2,NAICS3,Establishments,TOTAL10
count,101568.000000,57796.000000,1.056130e+05,1.056130e+05
mean,48.304397,481.598882,2.596430e+02,1.278511e+04
std,13.890403,131.237150,6.270791e+03,3.058448e+05
min,11.000000,111.000000,3.000000e+00,3.000000e+00
25%,42.000000,424.000000,6.000000e+00,1.380000e+02
50%,48.000000,459.000000,1.500000e+01,5.100000e+02
75%,55.000000,551.000000,5.100000e+01,2.166000e+03
max,81.000000,814.000000,1.585519e+06,7.784510e+07


In [8]:
EEOC1_2023.isnull().sum() *100/len(EEOC1_2023)

Nation        0.000000
Region        3.674737
Division      4.079043
State         4.962457
CBSA         15.482942
               ...    
HISPF1_2      0.000000
ASIANF1_2     0.000000
AIANF1_2      0.000000
NHOPIF1_2     0.000000
TOMRF1_2      0.000000
Length: 275, dtype: float64

In [9]:
EEOC1_2023['State'].info()

<class 'pandas.Series'>
RangeIndex: 105613 entries, 0 to 105612
Series name: State
Non-Null Count   Dtype
--------------   -----
100372 non-null  str  
dtypes: str(1)
memory usage: 825.2 KB


In [10]:
EEOC1_2023.dtypes

Nation       str
Region       str
Division     str
State        str
CBSA         str
            ... 
HISPF1_2     str
ASIANF1_2    str
AIANF1_2     str
NHOPIF1_2    str
TOMRF1_2     str
Length: 275, dtype: object

In [11]:
EEOC1_2023['TOTAL10'].dtypes
EEOC1_2023['tomrT10']

0         2233847
1            5456
2            1721
3            1302
4              48
           ...   
105608         77
105609         28
105610        101
105611         15
105612          *
Name: tomrT10, Length: 105613, dtype: str

In [12]:
EEOC1_2023['WHT1'].info()
EEOC1_2023['WHF10']

<class 'pandas.Series'>
RangeIndex: 105613 entries, 0 to 105612
Series name: WHT1
Non-Null Count   Dtype
--------------   -----
105613 non-null  str  
dtypes: str(1)
memory usage: 825.2 KB


0         20331221
1            36495
2            13365
3            15923
4              680
            ...   
105608        1216
105609         517
105610         361
105611          52
105612           8
Name: WHF10, Length: 105613, dtype: str

In [13]:
df = EEOC1_2023 #I got tired of typing out the full name

#Renaming Total_Race Variables
df.rename({
    'WHT10' : "White_Total",
    'BLKT10' : 'Black_Total',
    'HISPT10' : 'Hisp_Total',
    'ASIANT10' : 'Asian_Total',
    'AIANT10' : 'Amer_Ind_Total', 
    'NHOPIM10' : 'P_Isld_Total', 
    'tomrT10' : 'Multi_Race_Total'

}, axis=1, inplace=True)

In [14]:
#Renaming Total_Sex Variables
df.rename({
    'MT10' : 'Male_Total',
    'FT10': 'Female_Total'
}, axis=1, inplace=True)

In [15]:
#Renaming Job_Levels
df.rename({
    'TOTAL1': 'Senior_Managers', 
    'TOTAL2': 'Professionals',
    'TOTAL3': "Technicians",
    'TOTAL4': 'Sales_Workers',
    'TOTAL5': 'Clericals',
    'TOTAL6': 'Craft',
    'TOTAL7': 'Operatives',
    'TOTAL8': 'Labors',
    'TOTAL9': 'Mid_Managers'
}, axis= 1, inplace=True)

In [16]:
df['NAICS2_Name']

0                                                   NaN
1            Agriculture, Forestry, Fishing and Hunting
2            Agriculture, Forestry, Fishing and Hunting
3            Agriculture, Forestry, Fishing and Hunting
4            Agriculture, Forestry, Fishing and Hunting
                              ...                      
105608                Health Care and Social Assistance
105609                Health Care and Social Assistance
105610                  Accommodation and Food Services
105611    Other Services (except Public Administration)
105612    Other Services (except Public Administration)
Name: NAICS2_Name, Length: 105613, dtype: str

In [17]:
df['Year'] = 2023

#New Dataframe with Agg Columns Removed
df_cln = df[['Year', 'State', 'County', 'NAICS2_Name', 'Male_Total', 'Female_Total', 'White_Total','Black_Total', 'Hisp_Total',
            'Asian_Total', 'Amer_Ind_Total', 'P_Isld_Total', 'Multi_Race_Total',
            'Senior_Managers', 'Professionals', 'Technicians', 'Sales_Workers', 'Clericals', 'Craft', 'Operatives', 'Labors', 'Mid_Managers']]

df_cln.info()

<class 'pandas.DataFrame'>
RangeIndex: 105613 entries, 0 to 105612
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   Year              105613 non-null  int64
 1   State             100372 non-null  str  
 2   County            62419 non-null   str  
 3   NAICS2_Name       101568 non-null  str  
 4   Male_Total        105613 non-null  str  
 5   Female_Total      105613 non-null  str  
 6   White_Total       105613 non-null  str  
 7   Black_Total       105613 non-null  str  
 8   Hisp_Total        105613 non-null  str  
 9   Asian_Total       105613 non-null  str  
 10  Amer_Ind_Total    105613 non-null  str  
 11  P_Isld_Total      105613 non-null  str  
 12  Multi_Race_Total  105613 non-null  str  
 13  Senior_Managers   105613 non-null  str  
 14  Professionals     105613 non-null  str  
 15  Technicians       105613 non-null  str  
 16  Sales_Workers     105613 non-null  str  
 17  Clericals         105

In [18]:
#make sure all values in Male_Total Column are a number
df_cln = df_cln.replace(['*', ''], np.nan)

In [19]:
#Change Total Columns to Int
df_cln.astype({'Male_Total': 'Int64', 'Female_Total': 'Int64', 'White_Total': 'Int64','Black_Total': 'Int64', 'Hisp_Total': 'Int64',
            'Asian_Total': 'Int64', 'Amer_Ind_Total': 'Int64', 'P_Isld_Total': 'Int64', 'Multi_Race_Total': 'Int64',
            'Senior_Managers': 'Int64', 'Professionals': 'Int64', 'Technicians': 'Int64', 'Sales_Workers': 'Int64', 
            'Clericals': 'Int64', 'Craft': 'Int64', 'Operatives': 'Int64', 'Labors': 'Int64', 'Mid_Managers': 'Int64'

})

,Year,State,County,NAICS2_Name,Male_Total,Female_Total,White_Total,Black_Total,Hisp_Total,Asian_Total,...,Multi_Race_Total,Senior_Managers,Professionals,Technicians,Sales_Workers,Clericals,Craft,Operatives,Labors,Mid_Managers
0,2023,NaN,NaN,NaN,39671364,38173733,42741322,11986834,14457050,5476881,...,2233847,1228770,17053752,4221921,9531002,8148679,3865601,7094110,6105353,12453155
1,2023,NaN,NaN,"Agriculture, Forestry, Fishing and Hunting",270931,135100,113103,30723,246872,6300,...,5456,5505,18931,14307,10298,17116,16154,60072,225197,8774
2,2023,NaN,NaN,"Agriculture, Forestry, Fishing and Hunting",108316,54489,40388,5647,111664,2236,...,1721,2378,6615,4516,7510,7421,4441,18106,94367,5273
3,2023,NaN,NaN,"Agriculture, Forestry, Fishing and Hunting",72318,39664,45576,21697,39627,2333,...,1302,1628,7431,6977,618,4363,6279,28320,44509,1367
4,2023,NaN,NaN,"Agriculture, Forestry, Fishing and Hunting",4167,940,3212,302,1439,34,...,48,118,699,92,81,401,280,1029,1793,40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105608,2023,Washington,Yakima,Health Care and Social Assistance,869,3324,1687,51,2218,119,...,77,85,1501,279,8,771,39,11,26,1194
105609,2023,Washington,Yakima,Health Care and Social Assistance,301,1326,641,36,842,41,...,28,24,211,93,6,96,8,13,141,943
105610,2023,Washington,Yakima,Accommodation and Food Services,873,1121,603,34,1171,25,...,101,11,3,4,<NA>,14,<NA>,<NA>,30,1788
105611,2023,Washington,Yakima,Other Services (except Public Administration),226,167,152,4,202,10,...,15,<NA>,21,<NA>,49,64,34,87,81,12


In [20]:
# Save the dataframe to a new CSV file
df_cln.to_csv('EEOC_Data/EEOC_Cleaned_Data/EEOC1_2023_Cleaned.csv', index=False)

##### I used the `EEOC1_Exploration.ipynb` script to create a my first auto-cleaning script, `EEOC_Auto_Cleaner.ipynb` for the rest of the EEOC_1 datasets.
- Refer to `EEOC_Auto_Cleaner.ipynb` for the full script.

#### On 4/26/26 I discovered I was terribly wrong. 

By removing those aggregated columns I screwed myself. 
I realized that EEOC data is not microdata, it contains embeded sub-totals at multiple levels. I needed to start back from scratch and restore all of the columns I removed and renamed. 
This led to `EEOC_Auto_Cleaner_...` v2.0 and v3.0.

`EEOC_Auto_Cleaner_v3.0`:

```python

#This new version of the EEOC_Auto_Cleaner will now change the final file name and save it to the correct folder as well. 
#Same code but later data changed the way some varibales were named. Had to tweak the TOMRT10 Variable to all caps
import pandas as pd
import numpy as np
import os
import re

def EEOC_Auto_Clean(file_path, output_dir=r'C:\Users\jfraz\Code_You_Data_Analytics_Pathway\EEOC_Project\EEOC_Data\EEOC_Cleaned_Data'):

    df = pd.read_csv(file_path)
    
    # Renaming Total_Race Variables
    #df.rename({
     #   'WHT10' : "White_Total",
     #   'BLKT10' : 'Black_Total',
     #   'HISPT10' : 'Hisp_Total',
     #   'ASIANT10' : 'Asian_Total',
     #   'AIANT10' : 'Amer_Ind_Total', 
     #   'NHOPIT10' : 'P_Isld_Total',
     #   'nhopitT0' : 'P_Isld_Total',
     #   'TOMRT10' : 'Multi_Race_Total',
     #   'tomrT10' : 'Multi_Race_Total'
    #}, axis=1, inplace=True)

    # Renaming Total_Sex Variables
    df.rename({
        'MT10' : 'Male_Total',
        'FT10': 'Female_Total'
    }, axis=1, inplace=True)

    # Renaming Job_Levels
    df.rename({
        'TOTAL1': 'Senior_Managers', 
        'TOTAL2': 'Professionals',
        'TOTAL3': "Technicians",
        'TOTAL4': 'Sales_Workers',
        'TOTAL5': 'Clericals',
        'TOTAL6': 'Craft',
        'TOTAL7': 'Operatives',
        'TOTAL8': 'Labors',
        'TOTAL9': 'Mid_Managers'
    }, axis= 1, inplace=True)

    # Add a Year Column
    def extract_year_from_file_path(file_path):
        file_name = os.path.basename(file_path)
        year_pattern = r'(19|20)\d{2}'
        match = re.search(year_pattern, file_name)
        if match:
            return match.group()
        else:
            return "Unknown_Year"
        
    year_var = extract_year_from_file_path(file_path)
    df['Year'] = year_var

    # New Dataframe 
    df_cln = df.copy() # Added .copy() to prevent SettingWithCopyWarning
    

    # --- AUTOMATED SAVING PROCESS ---
    os.makedirs(output_dir, exist_ok=True)
    new_file_name = f"EEOC1_{year_var}_Cleaned.csv"
    full_export_path = os.path.join(output_dir, new_file_name)
    
    df_cln.to_csv(full_export_path, index=False)
    print(f"Successfully cleaned and saved: {full_export_path}")

    return df_cln

# --- Updated Execution Block ---

# 1. Define the folder containing all your raw EEOC CSV files
input_folder = r'c:\Users\jfraz\Code_You_Data_Analytics_Pathway\EEOC_Project\EEOC_Data\EEOC_1_Data'

# 2. Loop through every file found in that folder
for file_name in os.listdir(input_folder):
    
    # 3. Safety check: Only process files that end with .csv
    if file_name.endswith('.csv'):
        
        # 4. Combine the folder path and the file name to get the full path
        full_file_path = os.path.join(input_folder, file_name)
        
        # 5. Run your cleaning function on the file
        try:
            print(f"Processing: {file_name}...")
            EEOC_Auto_Clean(full_file_path)
        except Exception as e:
            # If a file is corrupted or formatted wrong, this stops the loop from crashing
            print(f"Error processing {file_name}: {e}")

print("Batch processing complete!")

```

### 4. Stacking EEOC_1 Tables 2014-2023

After cleaning each of the tables, I created a a script to stack all of them together into one table, `EEOC_DF_Merge.ipynb'.

Refer to `EEOC_DF_Merge.ipynb' for the actual script. Below is the script in markdown.

```python
import pandas as pd
from pathlib import Path
import os

def stack_csv(file_folder):
    folder_path = Path(file_folder)

    df_list = [pd.read_csv(file) for file in folder_path.glob('*.csv')]

    final_df = pd.concat(df_list, ignore_index=True)
    
    EEOC1_Master_Stacked = final_df

    EEOC1_Master_Stacked.to_csv(r'C:\Users\jfraz\Code_You_Data_Analytics_Pathway\EEOC_Project\EEOC_Data\EEOC1_Master_Stacked.csv', index=False)

    return EEOC1_Master_Stacked

my_folder = r'C:\Users\jfraz\Code_You_Data_Analytics_Pathway\EEOC_Project\EEOC_Data\EEOC_Cleaned_Data'

final_stack = stack_csv(my_folder)
print(final_stack.head())

df = pd.read_csv(r'c:\Users\jfraz\Code_You_Data_Analytics_Pathway\EEOC_Project\EEOC_Data\EEOC1_Master_Stacked.csv')

df.head()

df['Year'].unique()
df['Year'].value_counts()

#I finally cought the issue. the two or more race and Pacific islander variables were formatted differntly depending on the table year. 
#by making all variable names capitalized, It fixed the problem. 
import pandas as pd
from pathlib import Path

def stack_csv(file_folder):
    folder_path = Path(file_folder)
    df_list = []

    for file in folder_path.glob('*.csv'):
        # Read file and prevent mixed data type warnings
        temp_df = pd.read_csv(file, low_memory=False)
        
        # Remove hidden spaces and force uppercase for perfect column alignment
        temp_df.columns = temp_df.columns.str.strip()
        temp_df.columns = temp_df.columns.str.upper()
        
        df_list.append(temp_df)

    # Stack all cleaned dataframes together
    final_df = pd.concat(df_list, ignore_index=True)
    
    # Define the exact output path on your computer
    output_path = r'C:\Users\jfraz\Code_You_Data_Analytics_Pathway\EEOC_Project\EEOC_Data\EEOC1_Master_Stacked.csv'
    
    # Save the master file
    final_df.to_csv(output_path, index=False)

    return final_df

# Run the function
my_folder = r'C:\Users\jfraz\Code_You_Data_Analytics_Pathway\EEOC_Project\EEOC_Data\EEOC_Cleaned_Data'
final_stack = stack_csv(my_folder)

# Verify the results
print(final_stack.head())
```

### 5. Cleaning the EEOC1 Stacked Data Set

Now that I have one single Dataset with all of the years combined. I created another script to further clean it. I ended up making 3 iterations of the script, with the final saved as `EEOC1_Cleaning_Master_Stacked_v3.0`

Throughout the iterations:
- Version 1.0: Started making '.copy()' DataFrames so that when I messed up I would not have to keep loading the original DataFrame. 
    - Searched for duplicates.
    - Replaced the '*' values with NaN in the columns I wanted to make numeric. 
    - Isolated the Total Value Categrories by Year
    - Saved all of my New Data Frames as .csv files to the folder: `EEOC1_Cleaned_Master`
        - New DataFrames:
            - EEOC_Cleaned_Master_Stacked
            - EEOC1_CBSA_Total
            - EEOC1_County_Total
            - EEOC1_Demographic_Total
            - EEOC1_National_Baselines
            - EEOC1_State_Totals
- Version 2.0: Additionaly removed data that was causing double counting of values in the previously created datasets. Baselines will no longer overlap for Region, Division, CBSAs, etc. 
    - Saved the newly re-cleaned DataFrames to the folder: `EEOC1_Cleaned_Master_v2.0`
- Version 3.0: Created a new percentage (Pct) columns to each numerical column. 
    >**Note:** In 2022 the EEOC eliminated Type 6 reporting, causing a huge spkie in employment numbers for 2022-2023. In order to do Year over Year Analysis, the numeric columns needed to be changed from totals to percentages. 
    - Saved the newly re-cleaned DataFrames to the folder: `EEOC1_Cleaned_Master_v3.0`


Markdown version of `EEOC1_Cleaning_Master_Stacked_v3.0.ipynb` for the source file, please refer to the file as titled in the folder `Formatting_Scripts`.

```python
import pandas as pd
import dtale
import numpy as np

raw_df = pd.read_csv('EEOC_Data/EEOC1_Master_Stacked.csv')

# Created a copy file so when I mess up, I don't have to keep loading the original dataframe"
df = raw_df.copy()

df.info()

df.head()

#d = dtale.show(df)
#d.open_browser()

#Moving the 'Year' column to the first position on the dataframe
move_columns = df.pop('YEAR')
df.insert(0, 'YEAR', move_columns)
df.head()

#Replaceing the '*' values with NaN in the columns I want to make numeric. 
df.replace('*', np.nan, inplace=True)
#d = dtale.show(df)
#d.open_browser()

#I need to change all of my columns after 'Establishments' to numeric
#df.iloc[:, 11:] = pd.to_numeric(df.iloc[:, 11:], errors= 'coerce')
#df.dtypes
#df.info()

#The above cell didn't work. After researching, I'm trying this code instead
#df.iloc[:, 11:] = df.iloc[:, 11:].apply(pd.to_numeric, errors='coerce')
#df.dtypes
#df.info()

#previous code still did not work. Trying this AI corrected version instead. 
# 1. Grab the first 11 columns (leave them exactly as they are)
left_side = df.iloc[:, :11]

# 2. Grab column 11 onwards and convert them to numbers
right_side = df.iloc[:, 11:].apply(pd.to_numeric, errors='coerce')

# 3. Glue them back together side-by-side (axis=1) into a new dataframe
df = pd.concat([left_side, right_side], axis=1)

df.dtypes
df.info()

# 0. Set the Industry Baseline filter (Keep 2-digit NAICS, drop 3-digit sub-sectors)
industry_filter = df['NAICS2'].notna() & df['NAICS3'].isna()

# 1. Isolate the purely National totals (Every geographical subdivision must be blank)
df_national_baselines = df[
    industry_filter &
    df['REGION'].isna() &
    df['DIVISION'].isna() &
    df['STATE'].isna() &
    df['CBSA'].isna() &
    df['COUNTY'].isna()
].copy()

# 2. Isolate the purely State-level totals (Has State, but no deeper geography)
df_state_totals = df[
    industry_filter &
    df['STATE'].notna() &
    df['COUNTY'].isna() &
    df['CBSA'].isna()
].copy()

# 3. Create your main granular dataset for Counties (Has County, but no CBSA overlap)
df_county_totals = df[
    industry_filter &
    df['COUNTY'].notna() &
    df['CBSA'].isna()
].copy()

# 4. Create your main granular dataset for CBSAs (Has CBSA, but no County overlap)
df_CBSA_total = df[
    industry_filter &
    df['CBSA'].notna() &
    df['COUNTY'].isna()
].copy()

#d = dtale.show(df_national_baselines)
#d.open_browser()
#df.info()

#d = dtale.show(df_state_totals)
#d.open_browser()
#df.info()

#d = dtale.show(df_county_totals)
#d.open_browser()
#df.info()

#d = dtale.show(df_CBSA_total)
#d.open_browser()
#df.info()

Computer_Folder = 'EEOC_Data/EEOC1_Cleaned_Master/EEOC_Cleaned_Master_Stacked.csv'
df.to_csv(Computer_Folder, index=False)

import os

#Isolate the Total Value Categrories by Year
df_demographic_totals = df[df.iloc[:, 2:10].isna().all(axis=1)].copy()

#New code
def add_percentage_columns(df):
    """
    Takes a dataframe, finds all columns after TOTAL10, 
    calculates their percentage of TOTAL10, and appends them with a '_Pct' suffix.
    """
    # Safety check to ensure the column exists
    if 'TOTAL10' not in df.columns:
        return df
        
    # 1. Find the index of the 'TOTAL10' column 
    start_idx = df.columns.get_loc('TOTAL10') + 1
    
    # 2. Get the list of all demographic columns
    demographic_cols = df.columns[start_idx:]
    
    # 3. Divide all of those columns by TOTAL10 and multiply by 100
    df_pct = df[demographic_cols].div(df['TOTAL10'], axis=0) * 100
    
    # 4. Rename the new columns
    df_pct.columns = [f"{col}_Pct" for col in demographic_cols]
    
    # 5. Glue the new percentage columns onto the right side
    return pd.concat([df, df_pct], axis=1)

# Apply the function to the main master dataframe and all your filtered subsets
df = add_percentage_columns(df)
df_national_baselines = add_percentage_columns(df_national_baselines)
df_state_totals = add_percentage_columns(df_state_totals)
df_county_totals = add_percentage_columns(df_county_totals)
df_CBSA_total = add_percentage_columns(df_CBSA_total)
df_demographic_totals = add_percentage_columns(df_demographic_totals)

print("Percentage columns successfully added to all dataframes!")

#Saved all of my New Data Frames to CSV
folder_path = 'EEOC_Data/EEOC1_Cleaned_Master_v3.0'

master_file = os.path.join(folder_path, 'Master.csv')
national_file = os.path.join(folder_path, 'EEOC1_National_Baselines.csv')
state_file = os.path.join(folder_path, 'EEOC1_State_Totals.csv')
county_file = os.path.join(folder_path, 'EEOC1_County_Totals.csv')
cbsa_file = os.path.join(folder_path, 'EEOC1_CBSA_Total.csv')
demogr_file = os.path.join(folder_path, 'EEOC1_Demogr_Total.csv')

df.to_csv(master_file, index=False)
df_national_baselines.to_csv(national_file, index=False)
df_state_totals.to_csv(state_file, index=False)
df_county_totals.to_csv(county_file, index_label=False)
df_CBSA_total.to_csv(cbsa_file, index=False)
df_demographic_totals.to_csv(demogr_file, index=False)
```

# This Completes my Wrangling and Cleaning Journey!
The files created are what I used for my `EEOC1_EDA_Official.ipynb`